# Lab 05｜压缩以后，任务状态和原始依据各留下什么？

比较压缩前后的任务目标、纠正、行动约束和原始日志。再追问一个摘要未必保留的细节，观察 Agent 是否发现信息缺口并回查文件。完整 Thread 历史与当前模型输入分开查看。

## 核心代码：请求压缩，等完成后读取下一次模型输入

> 本讲重点看下面这 5 行。它们摘自后面的代码单元格，仅供定位；运行时仍从公共准备开始按顺序执行。单元格从标题所在的第 1 格起计数，行号按格内源码计算。

**第 2 节，第 7 个单元格，第 1 至 2 行：**

```python
thread.compact()
compaction_turn = wait_compaction(thread, before_turn_ids)
```

`compact()` 请求原生压缩；`wait_compaction()` 等待本次新增的压缩 Turn 完成，不能把接口返回直接当作压缩结束。

**第 3 节，第 9 个单元格，第 1 至 3 行：**

```python
after_start = len(lab.requests)
recap = run(thread, "先不要调用工具。依据当前保留的历史，简述任务编号、目标、已排除的猜测、下一步和操作限制；不确定的地方明确说明。")
after_request = lab.requests[after_start]
```

在同一 Thread 追问任务状态，并取出压缩后这次追问的首个请求，用它与压缩前输入比较。


## 公共准备｜必跑，代码可折叠

选择 **Python 3.12** 内核，保留仓库的 `examples/notebook_support.py`。下格会自动补装固定版本的实验依赖；如果当前内核已经导入其他版本，请安装后重启内核。无需先到 README 手动安装。

九讲统一读取 `DEEPSEEK_API_KEY`；未设置时弹出隐藏输入框。模型通过原生 Responses 使用 DeepSeek Flash，不需要 OpenAI / ChatGPT 账号；开始任务后会产生 API 费用。准备区只启动本地连接，不调用模型。每讲可从本格独立启动，最后一格关闭连接；再次从头运行前先执行清理。

版本检查、请求观察和退出处理属于辅助代码，不必逐行阅读。主流程以“必读必跑”标注；“扩展/可选”默认跳过。


In [ ]:
import sys, os, json, secrets, time, subprocess, getpass
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
from IPython.display import display, Markdown, HTML
from html import escape
from contextlib import redirect_stdout
from io import StringIO

if sys.version_info[:2] != (3, 12):
    raise RuntimeError("请选择 Python 3.12 内核，再从头运行。")
required = {"openai-codex": "0.154.0"}
missing = []
for package, expected in required.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    if installed != expected:
        missing.append(package + "==" + expected)
if missing:
    already_loaded = any(name in sys.modules for name in ("openai_codex",))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    if already_loaded:
        raise RuntimeError("依赖已安装；请重启内核并重新运行，避免混用旧模块。")

here = Path.cwd().resolve()
examples = next((p for base in (here, *here.parents)
                 for p in (base, base / "examples") if (p / "notebook_support.py").is_file()), None)
if examples is None:
    raise FileNotFoundError("请在完整仓库内打开本 notebook。")
sys.path.insert(0, str(examples))
from notebook_support import Lab, command_receipts, tool_output_contains
from openai_codex import ApprovalMode, Sandbox
if not os.environ.get("DEEPSEEK_API_KEY", "").strip():
    os.environ["DEEPSEEK_API_KEY"] = getpass.getpass("DEEPSEEK_API_KEY：").strip()
if "lab" in globals() and not globals()["lab"]._closed:
    raise RuntimeError("本讲连接仍在运行，请先执行最后的 lab.close()。")
lab = Lab(examples / "05-context-compression")

# 展示函数只整理真实材料，不把预期结果冒充本次运行结果。
def table(headers, rows):
    head = "".join(f"<th>{escape(str(v))}</th>" for v in headers)
    body = "".join("<tr>" + "".join(
        "<td><pre style='white-space:pre-wrap;max-width:52em'>" + escape(lab.clean(v)) + "</pre></td>"
        for v in row) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

def details(title, value):
    display(HTML("<details><summary>" + escape(title) + "</summary><pre style='white-space:pre-wrap'>"
                 + escape(lab.clean(value)) + "</pre></details>"))

def run(thread, prompt, **kwargs):
    # 保留运行错误；仅收起辅助层重复打印的状态和回答。
    with redirect_stdout(StringIO()):
        return lab.run(thread, prompt, **kwargs)

def events(result):
    rows = []
    for wrapped in result.items:
        item = wrapped.root
        if item.type == "commandExecution":
            rows.extend([("采取的动作", item.command),
                         ("刚获得的信息", item.aggregated_output or "（无文本输出）")])
        elif item.type == "fileChange":
            rows.append(("文件操作", ", ".join(change.path for change in item.changes)))
        elif item.type == "agentMessage" and item.text.strip():
            rows.append(("Agent 的判断", item.text))
    table(["事件", "本次实际内容"], rows)

def snapshot(folder):
    return {str(p.relative_to(folder)):p.read_bytes() for p in folder.rglob("*") if p.is_file()}

def compare_inputs(folder, before):
    after = snapshot(folder)
    changed = [name for name in sorted(before.keys() | after.keys()) if before.get(name) != after.get(name)]
    print("输入文件：" + ("有变化，展开下方查看" if changed else "未改变"))
    for name in changed:
        details(name, "运行前：\n" + before.get(name,"[不存在]".encode()).decode(errors="replace")
                + "\n运行后：\n" + after.get(name,"[不存在]".encode()).decode(errors="replace"))



WORK = lab.work
logfile = WORK / "capacity.log"
# 重复采样数据由准备区生成；原文只在 Agent 读文件时进入模型输入。
lines = [f"sample={i:04d} service=payment-api cpu_pct=24 memory_pct=41 queue_wait_ms=12 elapsed_us={1000+secrets.randbelow(9000)} state=healthy" for i in range(180)]
lines += ["02:07 pool_wait_ms=3600 active_connections=60 pool_limit=60",
          "任务编号：CASE-527；下一步：核对连接池配置变更；当前没有数据库故障的证据。"]
logfile.write_text("\n".join(lines),encoding="utf-8")
input_before = logfile.read_bytes()

def wait_compaction(thread, before_turn_ids):
    deadline = time.monotonic() + 240
    compaction_turn = None
    while time.monotonic() < deadline:
        current = thread.read(include_turns=True).thread
        candidates = [turn for turn in current.turns if turn.id not in before_turn_ids
                      and any(item.root.type == "contextCompaction" for item in turn.items)]
        if candidates and candidates[-1].status.value in ("completed", "failed", "interrupted"):
            compaction_turn = candidates[-1]
            break
        time.sleep(1)
    assert compaction_turn is not None, "240 秒内没有观察到压缩完成事件。"
    assert compaction_turn.status.value == "completed", "压缩未完成，不能继续声称已保留任务状态。"

    return compaction_turn

def input_text(request):
    return json.dumps(request.get("input",[]),ensure_ascii=False)

print("公共准备完成。")


## 1. Agent 先读取日志，再接受纠正｜必读必跑

日志的大部分是正常采样，末尾记录连接等待异常。目标只是拟定检查计划，不实施重启。下面由 Agent 自己调用工具读取日志，notebook 不替它投递内容。


In [ ]:
thread = lab.thread(developer_instructions="仅做只读调查，即使发现异常也不得重启服务。")
first = run(thread, "完整读取 capacity.log，为 CASE-527 拟定下一步检查计划，暂不作最终归因。概括值得注意的现象，不超过 150 字。")
correction_text = "补充确认：CPU 一直正常，排除 CPU 饱和的旧猜测。先核对连接池配置变更，不能重启服务。简短确认，不调用工具。"
corrected = run(thread, correction_text)
before_request = lab.requests[-1]
before_chars = len(input_text(before_request))
before_history = thread.read(include_turns=True).thread
before_turn_ids = {turn.id for turn in before_history.turns}
table(["压缩前的依据", "实际内容"], [
    ("Agent 读日志后的判断",first.final_response),
    ("随后收到的纠正",correction_text),
])
events(first)


## 2. 调用原生压缩，等完成事件｜必读必跑

`compact()` 只表示请求受理。辅助函数等到本次新增的压缩 Turn 完成，并确认存在 `contextCompaction` Item。这里调用实际压缩接口。


In [ ]:
thread.compact()
compaction_turn = wait_compaction(thread, before_turn_ids)
table(["压缩事件", "实际记录"], [
    ("Turn 状态", compaction_turn.status.value),
    ("Item", ", ".join(item.root.type for item in compaction_turn.items)),
])


## 3. 当前输入、完整历史、磁盘原文分开看｜必读必跑

先让 Agent 复述保留的任务状态，比较压缩前的依据与压缩后还记得的依据。字符数只表示输入长度，不等于 token 数或费用。

接着核对两条采样原文是否仍在当前输入、是否能从 `thread.read(include_turns=True)` 取得的完整历史找到。最后只提出需要的细节，观察 Agent 怎样补足信息；notebook 不向它发送原文，也不指定读取命令。若摘要已保留足够细节，或本次没有发生回读，如实看事件，不预设结果。

**继续观察**：CPU 饱和是否仍被排除？禁止重启是否保留？尚未证实的数据库故障有没有变成定论？读完下格再清理。


In [ ]:
after_start = len(lab.requests)
recap = run(thread, "先不要调用工具。依据当前保留的历史，简述任务编号、目标、已排除的猜测、下一步和操作限制；不确定的地方明确说明。")
after_request = lab.requests[after_start]
after_chars = len(input_text(after_request))
history_after = thread.read(include_turns=True).thread
history_json = history_after.model_dump_json(by_alias=True)
original_samples = [lines[73], lines[127]]
table(["保存位置", "采样 73 原文", "采样 127 原文"], [
    (name, *["可找到" if line in text else "未找到原句" for line in original_samples])
    for name,text in [("压缩前模型输入",input_text(before_request)),
                      ("压缩后模型输入",input_text(after_request)),
                      ("压缩后读取完整 Thread 历史",history_json),
                      ("磁盘原文件",logfile.read_text())]
])
table(["比较项", "压缩前", "压缩后"], [
    ("输入字符数",before_chars,after_chars),
    ("依据与任务状态",first.final_response + "\n" + correction_text,recap.final_response),
    ("压缩前 Turn 的记录",len(before_turn_ids),
     f"完整历史仍能取得 {len(before_turn_ids & {t.id for t in history_after.turns})} 条"),
])
reload_start = len(lab.requests)
recovered = run(thread, "请给出 sample=0073 与 sample=0127 的 elapsed_us 数值及二者差值。若现有依据不足，不要猜，请自行补足后再回答。")
events(recovered)
previous_calls = {r["call_id"] for r in command_receipts(lab.requests[:reload_start])}
new_reads = [r for r in command_receipts(lab.requests[reload_start:]) if r["call_id"] not in previous_calls]
details("扩展：本次追问新增的真实工具回执",json.dumps(new_reads,ensure_ascii=False,indent=2))
compare_inputs(WORK, {"capacity.log":input_before})


## 从本次观察带走什么

用目标、已排除项、未决问题与禁止动作检查任务状态，再用两条随机采样检查数值。摘要保留状态，原始来源提供回查；回查之后还需逐项核对数值和计算。输入变短与任务质量保持是两项独立验收。

### 把观察讲成设计判断

长调查既要控制输入体积，也要保留下一步行动。我优先保存目标、已排除猜测、下一步和禁止操作，让随机采样数值留在可回查原文中。压缩后先检查状态，再追问两个样本的差值，能暴露摘要缺项和原文恢复需求。摘要字符减少只说明文本变短；模型 token 成本和数值正确性仍要分别测量。

**追问：摘要丢了数值，是压缩失败吗？** 要看契约要求保存什么。允许原文回查时，任务状态完整且模型会查缺失数值，可以满足目标；原文不可访问或任务要求离线精确计算时，这种摘要就不够。

讲述前回查本次请求与回执；结果以本次运行为准，示例措辞不增加实跑覆盖。


In [ ]:
# 所有继续观察完成后必跑。
lab.close()
